# Tutorial 3: Retrieval-Augmented Generation (RAG)

*Level: Intermediate*

---
## What we'll build

You're building a chatbot that answers questions from your own documents (a product knowledge base, a set of research papers or an internal wiki). The problem with using a language model out of the box is that it answers from its training data. RAG fixes this: the model retrieves the most relevant documents from your collection first, passes them as context to a Large Language Model, which generates a grounded answer based strictly on the retrieved evidence. Less hallucinations and no stale knowledge. This is what we'll build in this tutorial: a full RAG pipeline over a set of scientific documents.

## The dataset: SciFact

We use **SciFact**, a scientific fact-checking dataset from [BEIR](https://huggingface.co/datasets/BeIR/scifact). It contains short scientific claims as queries paired with supporting or refuting biomedical texts.

To keep the ingestion fast, we build a focused knowledge base containing the documents that appear in the qrels test split. These documents are linked to at least one query by a human relevance judgment. This keeps the collection small and ensures every document in it is relevant to the kinds of questions we will be asking.

## What we'll use

- **FastEmbed:** Qdrant's lightweight embedding library.
- **Qdrant:** our vector database and search engine.
- **Anthropic:** Anthropic Python SDK to call Claude for answer generation


> **Note:** In this tutorial we only ingest a subset of the dataset to keep things fast. If you want to speed up execution or experiment with the full SciFact corpus, switch to a GPU runtime in Google Colab: Runtime → Change runtime type → T4 GPU.

---

## 0. Setup

We start by installing the required libraries:

- **fastembed**: Qdrant's lightweight embedding library.
- **qdrant-client**: the Python client for Qdrant. It allows you to interact with your Qdrant cluster directly from Python.
- **datasets**: Hugging Face's library for loading datasets.
- **anthropic**: the Anthropic Python SDK for calling Claude.
- **tqdm**: progress bars for long-running loops.

In [ ]:
# GPU optional: if you switched to a GPU runtime, replace the fastembed install below
# with these two lines instead:
# !pip install onnxruntime-gpu -i https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ -qqq
# !pip install fastembed-gpu qdrant-client datasets anthropic tqdm -qqq

!pip install fastembed==0.8.0 qdrant-client==1.19.0 datasets==5.0.0 anthropic==0.111.0 tqdm==4.68.3 -qqq

In [ ]:
import numpy as np
from tqdm import tqdm
from collections import defaultdict

from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams,
    PointStruct, SparseVector,
    Prefetch, RrfQuery, Rrf,
)
from fastembed import TextEmbedding, SparseTextEmbedding
import anthropic

## 1. Create a Qdrant Cluster and setup a client connection

If you do not already have a Qdrant cluster, follow these steps to create one:

- Register for a [Qdrant Cloud account](https://cloud.qdrant.io/) using your email, Google, or Github credentials.
- Under Create a Free Cluster, enter a cluster name and select your preferred cloud provider and region.
- Click Create Free Cluster.
- Copy the API key when prompted and store it somewhere safe as it won't be displayed again.
- Copy the Cluster Endpoint. It should look something like `https://xxx.cloud.qdrant.io`.


Create a free cluster at [cloud.qdrant.io](https://cloud.qdrant.io).  
In Colab: open **Secrets** (key icon on the left sidebar) and add:
- `QDRANT_URL` : your cluster endpoint e.g. `https://xyz.us-east4-0.gcp.cloud.qdrant.io`
- `QDRANT_API_KEY` : your API key

Next create a client connection to your Qdrant cluster

In [ ]:
from google.colab import userdata

QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("Connected. Collections:", [c.name for c in client.get_collections().collections])

Connected. Collections: ['nfcorpus_hybrid', 'scifact_hybrid', 'tutorial1_scifact-collection', 'tutorial2_scifact', 'nfcorpus', 'research-papers', 'treccovid', 'scifact_filtered', 'scifact_tutorial1', 'Tutorial1', 'fashion_products']


---

## 2. Load SciFact

We load three splits from Hugging Face:

- **corpus**: the biomedical texts we'll index
- **queries**: short scientific claims
- **qrels**: ground-truth relevance judgments

We concatenate `title + text` for each document. Titles in scientific abstracts are typically informative and keyword-rich, so including them alongside the abstract generally improves retrieval quality.

To keep ingestion fast, we will only ingest the documents that appear in the qrels.

In [ ]:
corpus_dataset = load_dataset("BeIR/scifact", "corpus", split="corpus")

doc_ids      = [str(doc["_id"]) for doc in corpus_dataset]
doc_titles   = [doc["title"]    for doc in corpus_dataset]
doc_texts    = [doc["text"]     for doc in corpus_dataset]
doc_passages = [(t + ". " + x).strip() if t else x for t, x in zip(doc_titles, doc_texts)]

print(f"Full corpus: {len(doc_ids)} documents")

queries_dataset = load_dataset("BeIR/scifact", "queries", split="queries")
queries = {str(q["_id"]): q["text"] for q in queries_dataset}

qrels_dataset = load_dataset("BeIR/scifact-qrels", split="test")
qrels_dict    = defaultdict(dict)
for row in qrels_dataset:
    qrels_dict[str(row["query-id"])][str(row["corpus-id"])] = row["score"]

# Filter corpus to documents referenced in qrels
qrel_doc_ids = set(
    doc_id
    for doc_dict in qrels_dict.values()
    for doc_id in doc_dict.keys()
)

mask         = [did in qrel_doc_ids for did in doc_ids]
doc_ids      = [x for x, m in zip(doc_ids,      mask) if m]
doc_titles   = [x for x, m in zip(doc_titles,   mask) if m]
doc_texts    = [x for x, m in zip(doc_texts,    mask) if m]
doc_passages = [x for x, m in zip(doc_passages, mask) if m]

print(f"Filtered corpus: {len(doc_ids)} documents")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

corpus/corpus-00000-of-00001.parquet:   0%|          | 0.00/4.47M [00:00<?, ?B/s]

Generating corpus split:   0%|          | 0/5183 [00:00<?, ? examples/s]

Full corpus: 5183 documents


...

train.tsv: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/919 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/339 [00:00<?, ? examples/s]

Filtered corpus: 283 documents


---

## 3. Load the models

For retrieval, we will use **hybrid search** combining dense and sparse retrieval.
Dense retrieval excels at capturing semantic similarity: it catches paraphrases and related concepts.

Sparse retrieval is used for keyword matching: it scores documents based on exact term overlap, making it naturally strong for jargon-heavy biomedical texts.

Combining both gives us the best of both worlds.

We load:

- `bge-base-en-v1.5`: a dense embedding model for semantic retrieval
- `BM25`: a lexical model for keyword-based sparse retrieval

> **Note:** `bge-base-en-v1.5` has a context window of 512 tokens. In this tutorial we do not chunk documents and tolerate any truncation that occurs. In production, verify that your document lengths fit within your model's context window.

In [ ]:
DENSE_MODEL_NAME  = "BAAI/bge-base-en-v1.5"
SPARSE_MODEL_NAME = "Qdrant/bm25"
DENSE_DIM         = 768

print("Loading dense model...")
# dense_model = TextEmbedding(model_name=DENSE_MODEL_NAME, providers=["CUDAExecutionProvider"]) #If you're using GPU
dense_model  = TextEmbedding(model_name=DENSE_MODEL_NAME)

print("Loading sparse model...")
sparse_model = SparseTextEmbedding(model_name=SPARSE_MODEL_NAME)

print("Models ready.")

Loading dense model...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading sparse model...


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Models ready.


---
## 4. Create the hybrid collection and ingest

We create our collection with two **named vectors**:

- **dense**: 768-dim vectors from `bge-base-en-v1.5` used for semantic retrieval.
- **sparse**: sparse vectors from `BM25` used for keyword matching.

Each point looks like this:

```python
PointStruct(
    id=0,
    vector={
        "dense":  [...],         # 768-dim float vector
        "sparse": SparseVector(
            indices=[23, 401, 1205, ...],  # non-zero token indices
            values= [0.4, 0.7, 0.2, ...]  # corresponding BM25 weights
        ),
    },
    payload={
        "doc_id": "...",
        "title":  "...",
        "text":   "..."
    }
)
```



In [ ]:
COLLECTION_NAME = "tutorial3_scifact"

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing '{COLLECTION_NAME}'")

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=DENSE_DIM, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    },
)
print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'tutorial3_scifact' created.


Define the ingestion function and run.

In [ ]:
def ingest(doc_ids, doc_titles, doc_texts, doc_passages, batch_size=64):
    total     = len(doc_ids)
    start_idx = client.count(COLLECTION_NAME).count

    if start_idx >= total:
        print(f"Already ingested {total} points.")
        return

    if start_idx > 0:
        print(f"Resuming from index {start_idx}")

    for i in tqdm(range(start_idx, total, batch_size), desc="Ingesting"):
        end      = min(i + batch_size, total)
        passages = doc_passages[i:end]

        dense_vecs  = [e.tolist() for e in dense_model.embed(passages)]
        sparse_embs = list(sparse_model.embed(passages))

        client.upsert(
            collection_name=COLLECTION_NAME,
            points=[
                PointStruct(
                    id=i + j,
                    vector={
                        "dense": dense_vecs[j],
                        "sparse": SparseVector(
                            indices=sparse_embs[j].indices.tolist(),
                            values=sparse_embs[j].values.tolist(),
                        ),
                    },
                    payload={
                        "doc_id": doc_ids[i + j],
                        "title":  doc_titles[i + j],
                        "text":   doc_texts[i + j],
                    },
                )
                for j in range(end - i)
            ], 
            wait=True,
        )

    print(f"Done. {client.count(COLLECTION_NAME).count} points ingested.")


ingest(doc_ids, doc_titles, doc_texts, doc_passages)



Ingesting:   0%|          | 0/5 [00:00<?, ?it/s]

Ingesting:  20%|██        | 1/5 [02:23<09:34, 143.55s/it]

Ingesting:  40%|████      | 2/5 [04:39<06:58, 139.33s/it]

Ingesting:  60%|██████    | 3/5 [06:58<04:38, 139.12s/it]

Ingesting:  80%|████████  | 4/5 [09:08<02:15, 135.38s/it]

Ingesting: 100%|██████████| 5/5 [10:02<00:00, 120.51s/it]

Done. 283 points ingested.


---
## 5. Retrieval

We retrieve candidates using Qdrant's Universal Query API: a single `query_points` call that runs two parallel **prefetches** (one dense, one sparse) and fuses the results server-side using Reciprocal Rank Fusion (RRF).

RRF combines results from multiple retrievers using rank position only, not raw scores. This matters because cosine similarities and BM25 scores live in completely different ranges and cannot be directly compared.

By default RRF in Qdrant assigns equal weight to each retriever, which can dilute the signal from the stronger one. On scientific claims, dense retrieval is the stronger signal. We therefore weight dense at 3 and sparse at 1, incorporating sparse rankings but at a lower influence, as they are less accurate on this type of query.

> **Note:** The right balance between dense and sparse depends on your data and query distribution. Tune them against your own evaluation set and pick the configuration with the best metrics rather than guessing.

In [ ]:
def retrieve(question, top_k=5, prefetch_k=30):
    dense_vec  = list(dense_model.query_embed([question]))[0].tolist()
    sparse_emb = list(sparse_model.query_embed([question]))[0]
    sparse_vec = SparseVector(
        indices=sparse_emb.indices.tolist(),
        values=sparse_emb.values.tolist(),
    )

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=dense_vec,  using="dense",  limit=prefetch_k),
            Prefetch(query=sparse_vec, using="sparse", limit=prefetch_k),
        ],
        query=RrfQuery(rrf=Rrf(weights=[3.0, 1.0])),
        limit=top_k,
        with_payload=True,
    ).points

    return hits

---

## 6. Generation

We use the Anthropic Python SDK to call Claude. We use `claude-sonnet-4-5` as our generation model. Feel free to experiment with other models or providers.

Add your API key to Colab secrets under `ANTHROPIC_API_KEY`.

In [ ]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
anthropic_client  = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

MODEL_NAME = "claude-sonnet-4-5"

### The system prompt

The system prompt is a set of instructions passed to the model before the conversation begins. It shapes how the model behaves throughout the interaction, what role it plays, what constraints it operates under, and what it should or should not do.

In our RAG pipeline, we instruct the model to answer strictly from the retrieved context and to cite which documents it draws from. This keeps answers verifiable and reduces the risk of hallucination.

In [ ]:
SYSTEM_PROMPT = """You are a scientific assistant. Answer the user's question based strictly on the provided abstracts.
Be concise and precise. If the abstracts do not contain enough information to answer the question, say so clearly.
Always indicate which abstract(s) you are drawing from in your answer."""


def build_context(hits):
    context_parts = []
    for i, hit in enumerate(hits, 1):
        title = hit.payload.get("title", "No title")
        text  = hit.payload.get("text", "")
        context_parts.append(f"[{i}] {title}\n{text}")
    return "\n\n".join(context_parts)


def generate(question, hits):
    context      = build_context(hits)
    user_message = f"Abstracts:\n{context}\n\nQuestion: {question}"

    response = anthropic_client.messages.create(
        model=MODEL_NAME,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text

---

## 7. The RAG pipeline

We wire retrieval and generation into a single function. For a given question it retrieves the top-5 documents, prints their titles so you can inspect what was retrieved, then passes them to the LLM and returns the generated answer.

In [ ]:
def rag(question, top_k=5):
    print(f"Question: {question}\n")

    hits = retrieve(question, top_k=top_k)

    print(f"Retrieved abstracts:")
    for i, hit in enumerate(hits, 1):
        print(f"  [{i}] {hit.payload.get('title', 'No title')}")
    print()

    answer = generate(question, hits)
    print(f"Answer:\n{answer}")
    print("\n" + "=" * 80 + "\n")
    return answer

---

## 8. Run some queries

It's time to test it out. We use actual claims from the SciFact dataset, which are guaranteed to have at least one relevant abstract in our collection.

In [ ]:
questions = [
    "1 in 5 million in UK have abnormal PrP positivity.",
    "APOE4 expression in iPSC-derived neurons increases AlphaBeta production and tau phosphorylation, delaying GABA neuron degeneration.",
    "5% of perinatal mortality is due to low birth weight.",
    "A deficiency of vitamin B12 increases blood levels of homocysteine.",
    "ALDH1 expression is associated with poorer prognosis in breast cancer.",
]

for q in questions:
    rag(q)

Question: 1 in 5 million in UK have abnormal PrP positivity.

Retrieved abstracts:
  [1] Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey
  [2] Self-harm in prisons in England and Wales: an epidemiological study of prevalence, risk factors, clustering, and subsequent suicide
  [3] Diabetes treatments and risk of amputation, blindness, severe kidney failure, hyperglycaemia, and hypoglycaemia: open cohort study in primary care
  [4] Gene–environment interactions in 7610 women with breast cancer: prospective evidence from the Million Women Study
  [5] Mosaic PPM1D mutations are associated with predisposition to breast and ovarian cancer

Answer:
This statement is **incorrect** based on the provided abstracts.

According to abstract [1], which reports on a large-scale survey of archived appendix samples in the UK:

- **16 out of 32,441** appendix samples tested positive for abnormal prion protein (PrP)
- This indicate

Try your own questions:

In [ ]:
your_question = input("Ask a scientific question: ")
rag(your_question)

Ask a scientific question: sleep quality facts
Question: sleep quality facts

Retrieved abstracts:
  [1] Cognitive behavioral therapy vs zopiclone for treatment of chronic primary insomnia in older adults: a randomized controlled trial.
  [2] A clinical approach to circadian rhythm sleep disorders.
  [3] The relation between past exposure to fine particulate air pollution and prevalent anxiety: observational cohort study
  [4] The stimulatory potency of T cell antigens is influenced by the formation of the immunological synapse.
  [5] Mental Health Conditions Among Patients Seeking and Undergoing Bariatric Surgery: A Meta-analysis.

Answer:
Based on the provided abstracts, here are key facts about sleep quality:

## Treatment Approaches for Poor Sleep Quality

**Cognitive Behavioral Therapy (CBT) vs. Medication:**
- In older adults with chronic primary insomnia, CBT showed superior outcomes compared to zopiclone (sleeping medication) in both short- and long-term management [1]
- CBT im

'Based on the provided abstracts, here are key facts about sleep quality:\n\n## Treatment Approaches for Poor Sleep Quality\n\n**Cognitive Behavioral Therapy (CBT) vs. Medication:**\n- In older adults with chronic primary insomnia, CBT showed superior outcomes compared to zopiclone (sleeping medication) in both short- and long-term management [1]\n- CBT improved sleep efficiency from 81.4% at pretreatment to 90.1% at 6-month follow-up, while zopiclone showed minimal change (82.3% to 81.9%) [1]\n- CBT patients spent significantly more time in slow-wave sleep (stages 3 and 4) and less time awake during the night [1]\n- Zopiclone did not differ significantly from placebo on most outcome measures [1]\n\n## Sleep Disorders Related to Circadian Rhythms\n\n**Circadian Rhythm Sleep Disorders:**\n- These are characterized by insomnia and excessive sleepiness due to misalignment between internal circadian timing and the 24-hour environmental cycle [2]\n- Maladaptive behaviors often play an impor

---

## 9. Takeaways and closing thoughts

- **Retrieval quality drives generation quality:** The LLM can only ground its answers in the context it receives. Irrelevant or noisy retrieval increases the risk of incorrect or hallucinated outputs. While prompt design helps guide behavior, it cannot compensate for poor retrieval quality, which is why retrieval design (chunking, embedding model choice,indexing, retrieval startegy etc.) is critical in any RAG system.

- **The system prompt is important, but not sufficient on its own:** A system prompt helps constrain behavior, but it does not guarantee strict grounding or reliability. Robust systems rely on a combination of high-quality retrieval, careful prompting, and **safeguards** such as filtering and output validation to improve both grounding and reliability.

- **The pipeline is modular by design:** Retrieval and generation are independent components. You can swap the embedding model, the ranking strategy, or the LLM without touching the rest. When answers are poor, you can isolate whether the issue is in retrieval (wrong documents surfaced) or generation (wrong synthesis from correct documents), and address each separately.

- **This is a starting point:** A production-grade RAG system would add query rewriting to handle ambiguous or underspecified questions, multi-stage retrieval with reranking, context compression, and systematic evaluation of retrieval quality. Beyond these improvements, RAG has evolved well past the simple retrieve-then-generate pattern we built here. Corrective RAG adds a verification step that detects gaps in the retrieved context and triggers additional retrieval when needed. Agentic RAG goes further, giving the model tools to plan multi-step retrieval strategies and dynamically decide when and what to retrieve. These architectures address the core limitation of basic RAG: a single retrieval pass does not always surface everything the model needs to answer reliably.